# Python垃圾回收机制## 学习目标- 理解引用计数的工作原理- 掌握循环引用的产生和解决方法- 理解标记清除和分代收集算法- 学会使用objgraph调试内存泄漏

## 一、引用计数（Reference Counting）### 1. 什么是引用计数？Python中一切皆对象，每个对象都有一个引用计数，记录有多少个指针指向它。当引用计数为0时，Python会自动释放该对象的内存。**核心原则**：引用计数为0 → 触发垃圾回收

In [ ]:
"""引用计数的基本示例"""import sysa = []  # 空列表# getrefcount本身会引入一次计数print(f"创建后引用计数：{sys.getrefcount(a)}")  # 2b = a  # b也指向同一个对象print(f"b = a后引用计数：{sys.getrefcount(a)}")  # 3c = bd = bprint(f"多引用后引用计数：{sys.getrefcount(a)}")  # 5

### 2. 引用计数与函数作用域

In [ ]:
"""局部变量在函数结束后自动回收"""import osimport psutildef show_memory_info(hint):pid = os.getpid()p = psutil.Process(pid)info = p.memory_full_info()memory = info.uss / 1024 / 1024print(f"{hint} memory used: {memory:.2f} MB")def func():show_memory_info("initial")a = [i for i in range(10000000)]  # 创建大列表show_memory_info("after a created")# 函数结束，a被销毁func()show_memory_info("finished")# 可以看到：函数结束后内存恢复到初始状态# 因为局部变量a的引用计数变为0，被垃圾回收

In [ ]:
"""全局变量不会被自动回收"""def func2():global aa = [i for i in range(10000000)]  # 声明为全局变量# func2()# show_memory_info("after global")# 函数结束后，全局变量a仍然存在，占用大量内存

### 3. 手动释放内存使用 `del` 删除引用，然后调用 `gc.collect()` 强制垃圾回收。

In [ ]:
"""手动触发垃圾回收"""import gcdef manual_gc():a = [i for i in range(10000000)]show_memory_info("after a created")del a  # 删除引用gc.collect()  # 强制垃圾回收show_memory_info("after gc")# manual_gc()

## 二、循环引用（Circular Reference）### 1. 问题：互相引用导致内存泄漏

In [ ]:
"""循环引用的陷阱"""def circular_ref():a = [i for i in range(10000000)]b = [i for i in range(10000000)]show_memory_info("after a, b created")a.append(b)  # a引用bb.append(a)  # b引用a → 形成循环引用# 函数结束后，a和b引用计数都不为0！# 内存不会被释放！# circular_ref()# show_memory_info("finished")# 发现内存没有被回收！因为互相引用导致引用计数不为0

### 2. 解决方案：调用gc.collect()

In [ ]:
"""使用gc.collect()解决循环引用"""import gcdef fix_circular():a = [i for i in range(10000000)]b = [i for i in range(10000000)]a.append(b)b.append(a)# fix_circular()# gc.collect()  # Python自动回收循环引用# show_memory_info("finished")# 使用gc.collect()后，内存被正确回收！

## 三、标记清除和分代收集### 1. 标记清除（Mark-Sweep）从根对象（root objects）出发，遍历所有可达对象并标记。未被标记的对象即为不可达对象，会被回收。Python只对**容器类对象**（list、dict、set等）执行标记清除，因为只有它们可能产生循环引用。

In [ ]:
"""标记清除的本质理解"""# 假设有以下引用关系：# a → list1 → list2#        ↑#        └── list3## 从a出发可以到达：list1、list2、list3 → 全部标记为存活# b → list4 (不受a管理)# 如果a被删除，list1/list2/list3都变为不可达# Python的标记清除算法会从根对象重新遍历# 未被标记的对象（list1/list2/list3）被回收print("标记清除算法：从根出发，标记所有可达对象，清除未标记对象")

### 2. 分代收集（Generational Collection）Python将所有对象分为**三代**（0代/1代/2代）：- 新创建的对象 → 第0代- 经过一次垃圾回收后仍然存活 → 移到第1代- 再次存活 → 移到第2代**思想**：新生的对象更有可能被回收，存活更久的对象有更高概率继续存活。这样可以节约计算量，提高性能。

## 四、调试内存泄漏### 使用objgraph工具- `objgraph.show_refs()`：生成引用关系图- `objgraph.show_backrefs()`：生成反向引用图（更详细）

In [ ]:
"""objgraph调试内存泄漏示例"""# 需要先安装：pip install objgraph graphviz# import objgrapha = [1, 2, 3]b = [4, 5, 6]a.append(b)b.append(a)# objgraph.show_refs([a])  # 可视化引用关系# objgraph.show_backrefs([a])  # 可视化反向引用# 可以看到两个list互相引用，定位内存泄漏

## 五、练习与自测### 练习题1：观察引用计数**题目**：编写程序，观察一个列表对象的引用计数变化。要求：创建列表、赋值给多个变量、作为函数参数传递，打印每次的引用计数。

In [ ]:
"""练习题1解答：观察引用计数"""import sysdef observe_refcount():a = [1, 2, 3]print(f"1. 创建后：{sys.getrefcount(a)}")b = aprint(f"2. b = a后：{sys.getrefcount(a)}")c = ad = aprint(f"3. 多个引用后：{sys.getrefcount(a)}")def inner(x):print(f"4. 函数内：{sys.getrefcount(x)}")inner(a)print(f"5. 函数返回后：{sys.getrefcount(a)}")observe_refcount()

### 练习题2：实现垃圾回收判定算法**题目**：输入一个有向图，给定起点和边，输出不可达的节点。**示例**：起点=0，边=[(0,1),(1,2),(2,0),(3,4)] → 不可达节点={3,4}

In [ ]:
"""练习题2解答：垃圾回收判定算法"""def find_unreachable(start, edges):"""找到不可达节点"""# 收集所有节点nodes = set()nodes.add(start)for u, v in edges:nodes.add(u)nodes.add(v)# 构建邻接表graph = {n: [] for n in nodes}for u, v in edges:graph[u].append(v)# DFS标记可达节点reachable = set()def dfs(node):reachable.add(node)for neighbor in graph[node]:if neighbor not in reachable:dfs(neighbor)dfs(start)# 不可达 = 所有节点 - 可达节点return nodes - reachable# 测试edges = [(0, 1), (1, 2), (2, 0), (3, 4)]print(f"不可达节点：{find_unreachable(0, edges)}")  # {3, 4}

## 六、知识总结### 重点记忆1. **引用计数**- 引用计数为0 → 触发垃圾回收- 引用计数只是充分非必要条件- 循环引用导致引用计数不为02. **循环引用**- 两个或多个对象互相引用- Python通过标记清除和分代收集来处理- 可手动调用`gc.collect()`3. **标记清除（Mark-Sweep）**- 从根对象出发，遍历并标记- 只处理容器类对象4. **分代收集**- 对象分为0/1/2代- 新对象更可能被回收- 长期存活对象移至更高代5. **调试工具**- `objgraph.show_refs()`：引用关系图- `objgraph.show_backrefs()`：反向引用图